# 00 · Visualise one clip with ground truth

Renders every frame of one clip as a video, with:
- **ground-truth boxes**, coloured by class and labelled with the class name;
- **ignored regions** as a translucent magenta overlay.

This checks that the labels line up with the images before we build anything on them.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from IPython.display import Video, display

# --- Config ---
DATA_ROOT = Path("../multiple-object-video-detection-hw-2-ucu-ai-2026")  # unpacked dataset
SPLIT = "train"           # "train" or "val"
CLIP = "uav0000281_00460_v"               # clip folder name, or None = first clip
FPS = 25                  # playback speed
OUT_WIDTH = 960           # downscale for a light video file (None = original size)
OUT_DIR = Path("../outputs/videos")

CLASSES = ["person", "bicycle", "car", "van", "truck", "tricycle", "bus", "motor"]
# One BGR colour per class
COLORS = [(0, 255, 0), (255, 255, 0), (0, 165, 255), (255, 0, 0),
          (0, 0, 255), (0, 255, 255), (128, 0, 255), (255, 128, 128)]
IGNORE_COLOR = (255, 0, 255)  # magenta

## Helpers: read labels and draw a frame

In [ ]:
def yolo_to_xyxy(cx, cy, w, h, W, H):
    """Normalised centre/size -> pixel corner coordinates."""
    return (int((cx - w / 2) * W), int((cy - h / 2) * H),
            int((cx + w / 2) * W), int((cy + h / 2) * H))

def read_txt(path):
    """Rows of floats from a label file; empty array if missing or empty."""
    if not path.exists() or path.stat().st_size == 0:
        return np.zeros((0, 5))
    return np.atleast_2d(np.loadtxt(path))

def draw_frame(img, labels, ignores):
    H, W = img.shape[:2]
    # Ignored regions: translucent fill + border
    overlay = img.copy()
    for cx, cy, w, h in ignores:
        x1, y1, x2, y2 = yolo_to_xyxy(cx, cy, w, h, W, H)
        cv2.rectangle(overlay, (x1, y1), (x2, y2), IGNORE_COLOR, -1)
    img = cv2.addWeighted(overlay, 0.3, img, 0.7, 0)
    for cx, cy, w, h in ignores:
        x1, y1, x2, y2 = yolo_to_xyxy(cx, cy, w, h, W, H)
        cv2.rectangle(img, (x1, y1), (x2, y2), IGNORE_COLOR, 2)

    # Objects: box + class name
    for cls, cx, cy, w, h in labels:
        cls = int(cls)
        x1, y1, x2, y2 = yolo_to_xyxy(cx, cy, w, h, W, H)
        color = COLORS[cls]
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 1)
        cv2.putText(img, CLASSES[cls], (x1, max(y1 - 3, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)
    return img

def add_header(img, text):
    cv2.rectangle(img, (0, 0), (img.shape[1], 28), (0, 0, 0), -1)
    cv2.putText(img, text, (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                (255, 255, 255), 1, cv2.LINE_AA)
    return img

## Pick a clip

In [ ]:
split_dir = DATA_ROOT / SPLIT
clips = sorted(p.name for p in (split_dir / "images").iterdir() if p.is_dir())
print(f"{len(clips)} clips in {SPLIT}/")
clip = CLIP or clips[0]
frames = sorted((split_dir / "images" / clip).glob("*.jpg"))
print(f"Clip: {clip} — {len(frames)} frames")

In [ ]:
def render(frame_path):
    """Load one frame with its labels and draw everything."""
    stem = frame_path.stem
    img = cv2.imread(str(frame_path))
    labels = read_txt(split_dir / "labels" / clip / f"{stem}.txt")
    ignores = read_txt(split_dir / "ignore" / clip / f"{stem}.txt")
    img = draw_frame(img, labels, ignores)
    return add_header(img, f"{clip}  frame {stem}  objects: {len(labels)}  ignored: {len(ignores)}")

# Preview the first frame at full resolution
first = render(frames[0])
plt.figure(figsize=(16, 9))
plt.imshow(cv2.cvtColor(first, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

## Render the whole clip to MP4

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / f"{SPLIT}_{clip}.mp4"

with imageio.get_writer(out_path, fps=FPS, codec="libx264", quality=6,
                        macro_block_size=1) as writer:
    for f in frames:
        img = render(f)

        # 1. Resize if width exceeds layout threshold
        if OUT_WIDTH and img.shape[1] > OUT_WIDTH:
            scale = OUT_WIDTH / img.shape[1]
            new_w = OUT_WIDTH
            new_h = int(img.shape[0] * scale)
        else:
            new_h, new_w = img.shape[0], img.shape[1]

        # 2. CRITICAL FIX: Ensure BOTH dimensions are strictly even numbers
        new_w = (new_w // 2) * 2
        new_h = (new_h // 2) * 2

        # 3. Perform the resize
        img = cv2.resize(img, (new_w, new_h))

        # 4. Pass the frame to imageio
        writer.append_data(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

print(f"Saved {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)")

Play it inline. The video is embedded in the notebook, so **clear this cell's output before committing**, or open the MP4 from `outputs/videos/` instead.

In [ ]:
display(Video(str(out_path), embed=True, width=OUT_WIDTH or 960))